In [0]:
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.gold;

use weather_openmeteo.gold;

-- Create the Gold schema if it does not exist
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.gold;

-- Create the KPIs target table with the specific data types
CREATE TABLE IF NOT EXISTS weather_openmeteo.gold.weather_kpis (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    temperature DOUBLE,
    avg_temperature_7d DOUBLE,
    alerts BOOLEAN
)
USING DELTA;

-- Calculate KPIs from the Silver table using Window functions
WITH gold_kpis AS (
    SELECT 
        date,
        latitude,
        longitude,
        city,
        temperature,
        
        -- Calculates the average temperature for the current day and the next 6 days (7 days total)
        AVG(temperature) OVER (
            PARTITION BY city, latitude, longitude
            ORDER BY CAST(date AS DATE)
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS avg_temperature_7d,
        
        -- Returns true if the temperature exceeds 102.0000, otherwise false
        CASE 
            WHEN temperature > 90.0000 THEN true 
            ELSE false 
        END AS alerts
    FROM weather_openmeteo.silver.weather_clean
)

-- Execute the Merge operation into the Gold table
MERGE INTO weather_openmeteo.gold.weather_kpis AS target
USING gold_kpis AS source
ON target.date = source.date 
   AND target.latitude = source.latitude 
   AND target.longitude = source.longitude

-- When the record already exists, update the KPIs and metrics
/*WHEN MATCHED THEN
  UPDATE SET
    target.city = source.city,
    target.temperature = source.temperature,
    target.avg_temperature_7d = CAST(source.avg_temperature_7d AS DOUBLE),
    target.alerts = source.alerts
*/
-- When the record is new, insert it into the Gold table
WHEN NOT MATCHED THEN
  INSERT (date, latitude, longitude, city, temperature, avg_temperature_7d, alerts)
  VALUES (source.date, source.latitude, source.longitude, source.city, source.temperature, CAST(source.avg_temperature_7d AS DOUBLE), source.alerts);




